---
title: "Simulation Setup and Demand Generation"
format:
    html:
        code-fold: false
---

## Overview

This notebook prepares the substrate for the courier simulation. It does three things. First it precomputes the full shortest path travel time matrix for the cleaned network, held entirely in memory, which turns every later routing query into an array lookup. Second it generates a synthetic 24 hour time series of delivery orders, with pickups drawn from the restaurant POIs and dropoff probabilities weighted by the population density feature attached to every node, shaped by lunch and dinner peak load multipliers. Third it summarizes the order volume and saves an Altair visualization of the temporal distribution to `outputs`. The notebook halts the pipeline here, before any courier assignment logic.

## Imports

In [1]:
import time
from pathlib import Path

import altair as alt
import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

# paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN = PROJECT_ROOT / "data" / "clean"
OUTPUTS = PROJECT_ROOT / "outputs"

RNG = np.random.default_rng(42)

# load clean data
G = ox.load_graphml(CLEAN / "dc_drive_clean.graphml")
pois = gpd.read_file(CLEAN / "dc_restaurants_clean.geojson")
node_tracts = pd.read_csv(CLEAN / "dc_nodes_tracts.csv")
print(f"Graph: {len(G.nodes):,} nodes, {len(G.edges):,} edges")
print(f"POIs: {len(pois):,}, node-tract rows: {len(node_tracts):,}")

Graph: 10,027 nodes, 26,938 edges
POIs: 2,043, node-tract rows: 10,027


## Shortest Path Matrices in Memory

The simulation will ask for travel times between arbitrary node pairs millions of times, so we precompute the full all-pairs shortest path travel time matrix once and hold it in memory. On a 10,027 node network the travel time matrix occupies about 0.4 GB as float32, which is trivial for the 64 GB of unified memory on this machine. We build a sparse adjacency matrix, taking the minimum travel time over parallel edges, and run Dijkstra from every source in compiled scipy code.

In [2]:
# sparse adjacency from the graph, minimum travel time over parallel edges
nodes = np.array(list(G.nodes), dtype=np.int64)
n = len(nodes)
idx = {int(v): i for i, v in enumerate(nodes)}

rows = []
for u, v, d in G.edges(data=True):
    rows.append((idx[int(u)], idx[int(v)], float(d["travel_time"])))
edf = pd.DataFrame(rows, columns=["u", "v", "tt"])
edf = edf.sort_values("tt").drop_duplicates(["u", "v"], keep="first")
A_tt = csr_matrix((edf["tt"], (edf["u"], edf["v"])), shape=(n, n))
print(f"Adjacency built: {A_tt.nnz:,} directed edges after collapsing parallels")

Adjacency built: 26,815 directed edges after collapsing parallels


In [3]:
# all pairs Dijkstra in chunks, results stored as float32
def all_pairs(A, label):
    M = np.empty((n, n), dtype=np.float32)
    t0 = time.time()
    for start in range(0, n, 2000):
        ind = np.arange(start, min(start + 2000, n))
        M[ind] = dijkstra(A, directed=True, indices=ind)
    print(f"{label}: {M.shape[0]:,} x {M.shape[1]:,} in {time.time() - t0:.0f}s ({M.nbytes / 1e9:.2f} GB in memory)")
    return M

TT = all_pairs(A_tt, "Travel time matrix")
print(f"Unreachable pairs: {np.isinf(TT).sum():,} (expected 0 on the strongly connected core)")
print(f"Mean pairwise travel time: {TT.mean() / 60:.2f} min")

Travel time matrix: 10,027 x 10,027 in 11s (0.40 GB in memory)
Unreachable pairs: 0 (expected 0 on the strongly connected core)
Mean pairwise travel time: 10.85 min


In [4]:
# persist the restaurant-origin rows for the courier simulation, seconds rounded to uint16
rest_nodes = pois["nearest_node"].astype(np.int64)
rest_nodes = rest_nodes[rest_nodes.isin(nodes)].unique()
rest_idx = np.array([idx[int(v)] for v in rest_nodes])
rest_tt = np.rint(TT[rest_idx]).astype(np.uint16)

MATRIX_OUT = CLEAN / "dc_ttmatrix_restaurants.npz"
np.savez_compressed(MATRIX_OUT, tt_s=rest_tt, rest_nodes=rest_nodes, all_nodes=nodes)
print(f"Unique restaurant nodes: {len(rest_nodes):,}")
print(f"Saved: {MATRIX_OUT.name} ({MATRIX_OUT.stat().st_size / 1e6:.1f} MB)")
print("The full matrix recomputes in under a minute, so only the restaurant rows are persisted")

Unique restaurant nodes: 898
Saved: dc_ttmatrix_restaurants.npz (13.1 MB)
The full matrix recomputes in under a minute, so only the restaurant rows are persisted


## Demand Model

Orders arrive as a nonhomogeneous Poisson process over the 24 hour day. A base hourly shape captures the ordinary rhythm of the day, and the lunch and dinner windows are scaled by explicit peak load multipliers, which are the levers the peak load analysis will later stress. Pickups are drawn uniformly from the restaurant POIs. Dropoff probabilities are proportional to the population density attached to each node in the cleaning notebook, so demand lands where people live and nodes on the National Mall or in industrial tracts receive effectively none.

In [5]:
# hourly arrival rates: base shape times peak multipliers
BASE_ORDERS_PER_HOUR = 450
LUNCH_HOURS = [11, 12, 13]
DINNER_HOURS = [18, 19, 20]
LUNCH_MULT = 3.5
DINNER_MULT = 4.5

base_shape = np.array([
    0.15, 0.08, 0.05, 0.03, 0.03, 0.08,
    0.30, 0.50, 0.70, 0.60, 0.80, 1.00,
    1.10, 1.00, 0.90, 0.70, 0.80, 1.20,
    1.00, 1.10, 1.00, 0.90, 0.70, 0.40,
])
mult = np.ones(24)
mult[LUNCH_HOURS] = LUNCH_MULT
mult[DINNER_HOURS] = DINNER_MULT
hourly_rate = BASE_ORDERS_PER_HOUR * base_shape * mult

rate_table = pd.DataFrame({"hour": range(24), "rate": hourly_rate.round(0).astype(int)})
print(rate_table.set_index("hour").T.to_string())
print(f"\nExpected orders per day: {hourly_rate.sum():,.0f}")

hour  0   1   2   3   4   5    6    7    8    9    10    11    12    13   14   15   16   17    18    19    20   21   22   23
rate  68  36  22  14  14  36  135  225  315  270  360  1575  1733  1575  405  315  360  540  2025  2228  2025  405  315  180

Expected orders per day: 15,174


In [6]:
# dropoff sampling weights proportional to node population density
w = node_tracts.set_index("osmid")["pop_density_km2"].reindex(nodes).fillna(0).to_numpy()
p_drop = w / w.sum()
print(f"Nodes with nonzero dropoff probability: {(p_drop > 0).sum():,} of {n:,}")
print(f"Top 1% of nodes hold {100 * np.sort(p_drop)[::-1][: n // 100].sum():.1f}% of dropoff probability")

Nodes with nonzero dropoff probability: 10,027 of 10,027
Top 1% of nodes hold 5.1% of dropoff probability


In [7]:
# draw the order table for one simulated day
counts = RNG.poisson(hourly_rate)
N = counts.sum()
hours = np.repeat(np.arange(24), counts)
time_s = hours * 3600 + RNG.uniform(0, 3600, N)

pick_rows = RNG.integers(0, len(pois), N)
pickup_nodes = pois["nearest_node"].astype(np.int64).to_numpy()[pick_rows]
drop_nodes = RNG.choice(nodes, size=N, p=p_drop)

# resample the rare case where an order drops off at its own pickup node
clash = pickup_nodes == drop_nodes
while clash.any():
    drop_nodes[clash] = RNG.choice(nodes, size=clash.sum(), p=p_drop)
    clash = pickup_nodes == drop_nodes

pi = np.array([idx[int(v)] for v in pickup_nodes])
di = np.array([idx[int(v)] for v in drop_nodes])
orders = pd.DataFrame({
    "time_s": np.sort(time_s).round(1),
    "hour": hours[np.argsort(time_s)],
    "pickup_node": pickup_nodes[np.argsort(time_s)],
    "dropoff_node": drop_nodes[np.argsort(time_s)],
    "drive_time_s": TT[pi, di][np.argsort(time_s)].round(1),
})
orders.insert(0, "order_id", np.arange(1, N + 1))

ORDERS_OUT = CLEAN / "dc_orders_sim.csv"
orders.to_csv(ORDERS_OUT, index=False)
print(f"Orders generated: {N:,}")
print(f"Saved: {ORDERS_OUT.name} ({ORDERS_OUT.stat().st_size / 1e6:.1f} MB)")
print(orders.head(8).to_string(index=False))

Orders generated: 15,188
Saved: dc_orders_sim.csv (0.6 MB)
 order_id  time_s  hour  pickup_node  dropoff_node  drive_time_s
        1    81.8     0     49743231     646216520   1133.400024
        2   110.9     0    774340109      49779846    349.299988
        3   149.8     0     49793773      49839482    832.200012
        4   209.9     0     49745240      49745636    154.500000
        5   304.0     0   4807605970      49800282    346.000000
        6   315.5     0    641939501      49795559    591.900024
        7   324.2     0     49780840      49859684     28.500000
        8   347.0     0     49780562      49838680    459.500000


## Temporal Distribution

In [8]:
# altair chart of orders per 15 minute bin with the peak windows shaded
bins = (orders["time_s"] // 900).astype(int)
per_bin = bins.value_counts().sort_index().reindex(range(96), fill_value=0)
chart_df = pd.DataFrame({
    "time": pd.to_datetime(per_bin.index * 900, unit="s"),
    "orders": per_bin.values,
})
windows = pd.DataFrame({
    "start": pd.to_datetime([LUNCH_HOURS[0] * 3600, DINNER_HOURS[0] * 3600], unit="s"),
    "end": pd.to_datetime([(LUNCH_HOURS[-1] + 1) * 3600, (DINNER_HOURS[-1] + 1) * 3600], unit="s"),
    "window": [f"lunch x{LUNCH_MULT}", f"dinner x{DINNER_MULT}"],
})

shade = alt.Chart(windows).mark_rect(opacity=0.15).encode(
    x="start:T", x2="end:T", color=alt.Color("window:N", scale=alt.Scale(range=["#d7191c", "#2c7bb6"]), title="peak window"),
)
area = alt.Chart(chart_df).mark_area(color="#e07b54", opacity=0.8, line={"color": "#c0392b"}).encode(
    x=alt.X("time:T", axis=alt.Axis(format="%H:%M", title="time of day")),
    y=alt.Y("orders:Q", title="orders per 15 min"),
)
chart = (shade + area).properties(
    width=760, height=320,
    title=f"Simulated Delivery Demand over 24 Hours (n = {N:,} orders)",
)
CHART_OUT = OUTPUTS / "11_order_temporal_distribution.png"
chart.save(str(CHART_OUT), scale_factor=2.0)
print(f"Saved: {CHART_OUT.name}")
chart

Saved: 11_order_temporal_distribution.png


alt.LayerChart(...)

As we can see from the chart above, the simulated day has the twin peak profile that the meal delivery literature treats as the defining operational challenge (Reyes et al. 2018). The dinner window is the taller of the two, carrying 41.8 percent of the day's 15,188 orders against 31.7 percent for lunch, and the busiest 15 minute bin lands at 579 orders shortly after 19:00. Between the peaks the system idles at a small fraction of peak load, which is exactly the supply and demand imbalance that assignment strategy and fleet sizing will have to manage.

## Order Volume Summary

In [9]:
# phase 5 summary report
lunch_share = orders["hour"].isin(LUNCH_HOURS).mean()
dinner_share = orders["hour"].isin(DINNER_HOURS).mean()
peak_hour = orders["hour"].value_counts().idxmax()
peak_15 = per_bin.max()

print("=" * 62)
print("PHASE 5 SUMMARY: SIMULATED ORDER VOLUME")
print("=" * 62)
print(f"\nOrders in the simulated day: {N:,}")
print(f"Lunch window {LUNCH_HOURS[0]}:00 to {LUNCH_HOURS[-1] + 1}:00 (x{LUNCH_MULT}): {orders['hour'].isin(LUNCH_HOURS).sum():,} orders ({100 * lunch_share:.1f}%)")
print(f"Dinner window {DINNER_HOURS[0]}:00 to {DINNER_HOURS[-1] + 1}:00 (x{DINNER_MULT}): {orders['hour'].isin(DINNER_HOURS).sum():,} orders ({100 * dinner_share:.1f}%)")
print(f"Peak hour: {peak_hour}:00 with {orders['hour'].value_counts().max():,} orders")
print(f"Peak 15 minute bin: {peak_15:,} orders")
print(f"\nUnassisted drive time per order: mean {orders['drive_time_s'].mean() / 60:.2f} min, median {orders['drive_time_s'].median() / 60:.2f} min, p95 {orders['drive_time_s'].quantile(0.95) / 60:.2f} min")
print(f"\nMatrix in memory: travel time, {TT.nbytes / 1e9:.2f} GB, lookups are O(1)")
print(f"Persisted for the simulation: {MATRIX_OUT.name}, {ORDERS_OUT.name}")
print("\nPipeline halted after Phase 5 as instructed, courier assignment begins in notebook 06")
print("=" * 62)

PHASE 5 SUMMARY: SIMULATED ORDER VOLUME

Orders in the simulated day: 15,188
Lunch window 11:00 to 14:00 (x3.5): 4,810 orders (31.7%)
Dinner window 18:00 to 21:00 (x4.5): 6,341 orders (41.8%)
Peak hour: 19:00 with 2,279 orders
Peak 15 minute bin: 579 orders

Unassisted drive time per order: mean 8.26 min, median 7.79 min, p95 15.96 min

Matrix in memory: travel time, 0.40 GB, lookups are O(1)
Persisted for the simulation: dc_ttmatrix_restaurants.npz, dc_orders_sim.csv

Pipeline halted after Phase 5 as instructed, courier assignment begins in notebook 06


The substrate for the courier simulation is now complete. The order table pairs 898 unique restaurant pickup nodes with population weighted dropoffs across all 10,027 intersections, and each order is already annotated with its unassisted shortest path drive time (mean 8.26 minutes, p95 just under 16) from the in-memory matrix. Those numbers are the physical floor that no assignment strategy can beat, which makes them the natural benchmark for the strategy comparison in notebook 06. The pipeline halts here as instructed.